In [1]:
from pathlib import Path 

train_path = Path("../data/raw/train_data.csv")
test_path = Path("../data/raw/test_data.csv")

print("Training file found:", train_path.exists())
print("Test file found:", test_path.exists())

Training file found: True
Test file found: True


In [2]:
import pandas as pd

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("files loaded successfully")

files loaded successfully


In [3]:
print("Training data shape:", train_df.shape)
print("Testing data shape:", test_df.shape)
print("Total conversations:", len(train_df) + len(test_df))

Training data shape: (450, 8)
Testing data shape: (900, 8)
Total conversations: 1350


In [4]:
print("Training columns:")
print(train_df.columns.tolist())

print("\nTesting columns:")
print(test_df.columns.tolist())

Training columns:
['conversation', 'trait', 'attack_success', 'rationale', 'scenario_class', 'information_class', 'attack_strategies', 'label']

Testing columns:
['conversation', 'trait', 'attack_success', 'rationale', 'scenario_class', 'information_class', 'attack_strategies', 'label']


## Dataset fields

- **conversation:** The complete multi-turn conversation.
- **trait:** Simulated victim personality trait.
- **attack_success:** Success level of the malicious attempt.
- **rationale:** Explanation for the attack-success annotation.
- **scenario_class:** Funding agency, journalist, or recruiter scenario.
- **information_class:** Type of sensitive information involved.
- **attack_strategies:** Annotated manipulation strategies.
- **label:** Malicious or benign classification.

In [5]:
label_counts = pd.DataFrame({
    "Training": train_df["label"].value_counts(),
    "Testing": test_df["label"].value_counts()
}).fillna(0).astype(int)

label_counts["Total"] = (
    label_counts["Training"] + label_counts["Testing"]
)

label_counts = label_counts.rename(index={
    1.0: "Malicious",
    0.0: "Benign"
})

display(label_counts)

,Training,Testing,Total
label,,,
Malicious,300,600,900
Benign,150,300,450


## Class distribution

The dataset contains 900 malicious conversations and 450 benign conversations. Both the training and testing files maintain a 2:1 malicious-to-benign ratio. Because the classes are imbalanced, the final models will be evaluated using precision, recall, F1-score, false-positive rate, and false-negative rate in addition to accuracy.

In [6]:
scenario_counts = pd.DataFrame({
    "Training": train_df["scenario_class"].value_counts(),
    "Testing": test_df["scenario_class"].value_counts()
}).fillna(0).astype(int)

scenario_counts["Total"] = (
    scenario_counts["Training"] + scenario_counts["Testing"]
)

scenario_counts = scenario_counts.rename(index={
    "AF": "Funding agency",
    "JO": "Journalist",
    "RE": "Recruiter"
})

display(scenario_counts)

,Training,Testing,Total
scenario_class,,,
Funding agency,150,300,450
Journalist,150,300,450
Recruiter,150,300,450


## Scenario distribution

SE-VSim contains three professional social-engineering scenarios: funding agency, journalist, and recruiter interactions. Each scenario contains 450 conversations, with 150 in the training file and 300 in the testing file. The equal scenario distribution reduces the likelihood that one professional context will dominate the experiment.

In [7]:
information_counts = pd.DataFrame({
    "Training": train_df["information_class"].value_counts(),
    "Testing": test_df["information_class"].value_counts()
}).fillna(0).astype(int)

information_counts["Total"] = (
    information_counts["Training"] + information_counts["Testing"]
)

display(information_counts)

,Training,Testing,Total
information_class,,,
Patents and Trademarks,174,276,450
Personal Identifiable Information (PII),141,309,450
Sensitive Financial Information,135,315,450


## Information category distribution

The dataset contains three targeted information categories: patents and trademarks, personally identifiable information, and sensitive financial information. Each category contains 450 conversations overall. However, their distributions differ between the supplied training and testing files. This difference should be considered when creating the project's final conversation-level training, validation, and testing partitions.

In [8]:
missing_values = pd.DataFrame({
    "Training": train_df.isna().sum(),
    "Testing": test_df.isna().sum()
})

missing_values["Total"] = (
    missing_values["Training"] + missing_values["Testing"]
)

display(missing_values)

,Training,Testing,Total
conversation,0,0,0
trait,0,0,0
attack_success,150,300,450
rationale,150,300,450
scenario_class,0,0,0
information_class,0,0,0
attack_strategies,150,300,450
label,0,0,0


In [9]:
combined_df = pd.concat(
    [train_df, test_df],
    ignore_index=True
)


missing_metadata_by_label = combined_df.groupby("label")[
    ["attack_success", "rationale", "attack_strategies"]
].apply(lambda group: group.isna().sum())

display(missing_metadata_by_label)

,attack_success,rationale,attack_strategies
label,,,
0.0,450,450,450
1.0,0,0,0


## Missing values

The conversation, trait, scenario, information category, and label fields contain no missing values. The attack-success, rationale, and attack-strategy fields contain 450 missing values, corresponding to the benign conversations. These fields describe malicious attack characteristics and therefore do not apply to benign records. They will be excluded from model inputs rather than filled or imputed.

In [10]:
training_duplicates = train_df.duplicated().sum()
testing_duplicates = test_df.duplicated().sum()

print("Duplicate training rows:", training_duplicates)
print("Duplicate testing rows:", testing_duplicates)
print(
    "Total duplicate rows:",
    training_duplicates + testing_duplicates
)

Duplicate training rows: 0
Duplicate testing rows: 0
Total duplicate rows: 0


In [11]:
overlapping_conversations = set(train_df["conversation"]).intersection(
    set(test_df["conversation"])
)

print(
    "Conversations appearing in both files:",
    len(overlapping_conversations)
)

Conversations appearing in both files: 0


## Duplicate and overlap check

No complete duplicate rows were found in either dataset file. The conversation text was also compared between the supplied training and testing files to identify possible overlap. Preventing the same conversation from appearing in multiple partitions is necessary to avoid data leakage during model evaluation.

In [12]:
import ast

train_lengths = train_df["conversation"].apply(
    lambda conversation: len(ast.literal_eval(conversation))
)

test_lengths = test_df["conversation"].apply(
    lambda conversation: len(ast.literal_eval(conversation))
)

print("Training conversation lengths:")
display(train_lengths.value_counts().sort_index())

print("Testing conversation lengths:")
display(test_lengths.value_counts().sort_index())

Training conversation lengths:


conversation
11    450
Name: count, dtype: int64

Testing conversation lengths:


conversation
11    900
Name: count, dtype: int64

In [13]:
conversation_length_summary = pd.DataFrame({
    "Dataset": ["Training", "Testing"],
    "Minimum": [train_lengths.min(), test_lengths.min()],
    "Maximum": [train_lengths.max(), test_lengths.max()],
    "Average": [train_lengths.mean(), test_lengths.mean()]
})

display(conversation_length_summary)

,Dataset,Minimum,Maximum,Average
0,Training,11,11,11.0
1,Testing,11,11,11.0


## Conversation lengths

Every training and testing conversation contains 11 ordered entries. This consistent length simplifies turn-level conversion, but the ordering must still be preserved when individual messages are extracted. All turns derived from the same conversation must remain together during later data partitioning to prevent leakage.

In [14]:
from collections import Counter
import ast

def count_entry_types(dataframe):
    entry_types = Counter()

    for conversation_text in dataframe["conversation"]:
        conversation = ast.literal_eval(conversation_text)

        for entry in conversation:
            entry_types[type(entry).__name__] += 1

    return entry_types

print("Training entry types:")
print(count_entry_types(train_df))

print("\nTesting entry types:")
print(count_entry_types(test_df))

Training entry types:
Counter({'dict': 4850, 'str': 95, 'list': 5})

Testing entry types:
Counter({'dict': 9664, 'str': 236})


In [15]:
def count_irregular_conversations(dataframe):
    irregular_count = 0

    for conversation_text in dataframe["conversation"]:
        conversation = ast.literal_eval(conversation_text)

        if any(not isinstance(entry, dict) for entry in conversation):
            irregular_count += 1

    return irregular_count

print(
    "Training conversations with irregular entries:",
    count_irregular_conversations(train_df)
)

print(
    "Testing conversations with irregular entries:",
    count_irregular_conversations(test_df)
)

Training conversations with irregular entries: 47
Testing conversations with irregular entries: 81


## Internal conversation format

Most conversation entries are dictionaries containing message information. However, some entries are stored as strings or nested lists instead of consistent dictionary objects. These irregular entries must be normalized before the conversations can be converted into turn-level records. They will not be deleted automatically because their message information may still be recoverable.

In [16]:
def count_dictionary_keys(dataframe):
    key_counts = Counter()

    for conversation_text in dataframe["conversation"]:
        conversation = ast.literal_eval(conversation_text)

        for entry in conversation:
            if isinstance(entry, dict):
                key_counts[tuple(sorted(entry.keys()))] += 1

    return key_counts

print("Training dictionary structures:")
print(count_dictionary_keys(train_df))

print("\nTesting dictionary structures:")
print(count_dictionary_keys(test_df))

Training dictionary structures:
Counter({('Message', 'Name'): 4850})

Testing dictionary structures:
Counter({('Message', 'Name'): 9664})


In [17]:
def inspect_dictionary_entries(dataframe):
    results = {
        "dictionary_entries": 0,
        "missing_name": 0,
        "missing_message": 0,
        "blank_name": 0,
        "blank_message": 0
    }

    for conversation_text in dataframe["conversation"]:
        conversation = ast.literal_eval(conversation_text)

        for entry in conversation:
            if isinstance(entry, dict):
                results["dictionary_entries"] += 1

                if "Name" not in entry:
                    results["missing_name"] += 1
                elif not str(entry["Name"]).strip():
                    results["blank_name"] += 1

                if "Message" not in entry:
                    results["missing_message"] += 1
                elif not str(entry["Message"]).strip():
                    results["blank_message"] += 1

    return results

print("Training dictionary inspection:")
print(inspect_dictionary_entries(train_df))

print("\nTesting dictionary inspection:")
print(inspect_dictionary_entries(test_df))

Training dictionary inspection:
{'dictionary_entries': 4850, 'missing_name': 0, 'missing_message': 0, 'blank_name': 0, 'blank_message': 0}

Testing dictionary inspection:
{'dictionary_entries': 9664, 'missing_name': 0, 'missing_message': 0, 'blank_name': 0, 'blank_message': 0}


## Standard message structure

The correctly formatted conversation entries use two fields: `Name`, which identifies the message sender, and `Message`, which contains the message text. The standard dictionary entries were checked for missing and blank names and messages. Irregular string and list entries will be handled separately during normalization.

In [18]:
combined_df = pd.concat(
    [train_df, test_df],
    ignore_index=True
)

inspection_summary = pd.DataFrame({
    "Measure": [
        "Training conversations",
        "Testing conversations",
        "Total conversations",
        "Malicious conversations",
        "Benign conversations",
        "Funding agency conversations",
        "Journalist conversations",
        "Recruiter conversations",
        "Minimum entries per conversation",
        "Maximum entries per conversation",
        "Duplicate training rows",
        "Duplicate testing rows",
        "Overlapping conversations",
        "Irregular training conversations",
        "Irregular testing conversations"
    ],
    "Value": [
        len(train_df),
        len(test_df),
        len(combined_df),
        int((combined_df["label"] == 1).sum()),
        int((combined_df["label"] == 0).sum()),
        int((combined_df["scenario_class"] == "AF").sum()),
        int((combined_df["scenario_class"] == "JO").sum()),
        int((combined_df["scenario_class"] == "RE").sum()),
        int(min(train_lengths.min(), test_lengths.min())),
        int(max(train_lengths.max(), test_lengths.max())),
        int(train_df.duplicated().sum()),
        int(test_df.duplicated().sum()),
        len(overlapping_conversations),
        count_irregular_conversations(train_df),
        count_irregular_conversations(test_df)
    ]
})

display(inspection_summary)

,Measure,Value
0,Training conversations,450
1,Testing conversations,900
2,Total conversations,1350
3,Malicious conversations,900
4,Benign conversations,450
5,Funding agency conversations,450
6,Journalist conversations,450
7,Recruiter conversations,450
8,Minimum entries per conversation,11
9,Maximum entries per conversation,11


## Initial inspection conclusion

The SE-VSim files were successfully loaded and inspected. The dataset contains 1,350 synthetic multi-turn conversations: 900 malicious and 450 benign. The funding agency, journalist, and recruiter scenarios are evenly represented, and each conversation contains 11 ordered entries.

No complete duplicate rows or overlapping conversations were identified between the supplied training and testing files. The main data-quality issue is inconsistent formatting within some conversations. Most message entries are dictionaries containing `Name` and `Message`, but some entries are stored as strings or nested lists. The next implementation stage will normalize these entries and convert each conversation into ordered turn-level records without using personality, attack-success, rationale, or attack-strategy metadata as model inputs.